### Ridge Regression (L2 Regularization)

Normal Linear Regression, sadece hatayı minimize etmeye çalışır:
- Loss = Hata (Σ(y-ŷ)²)

Ridge Regression, buna bir ceza terimi (penalty) ekler:
- Loss = Hata + λ × Σ(w²)

- λ (lambda, sklearn'de "alpha" olarak geçer) → cezanın gücünü kontrol eden bir katsayı
- Σ(w²) → tüm katsayıların karelerinin toplamı (L2 cezası)

### Neden İşe Yarıyor?

- Model artık sadece "hatayı azalt" değil, aynı zamanda "katsayıları da küçük tut" baskısı altında.

- Bir modelin, veri setindeki gürültülü/sapan noktaları da yakalayabilmesi için genelde büyük ve agresif katsayılara ihtiyacı vardır çünkü eğrinin aniden kıvrılıp bu noktalara değebilmesi için büyük w değerleri gerekir. Ridge, büyük katsayıları cezalandırarak modelin bu agresif/kıvrımlı eğriyi çizmesini (yani gürültüyü ezberlemesini) engeller. Bu da modeli daha "sakin" ve genellenebilir hale getirir.

### Lambda'nın (alpha) Etkisi
- λ = 0 → Ridge, normal Linear Regression'a döner (ceza yok)
- λ küçük → hafif düzenlileştirme (regularization)
- λ büyük → katsayılar sıfıra yaklaşır, model çok basitleşir (underfitting riski)

Sonuç olarak; λ arttıkça model daha basit/sakin hale gelir → variance azalır ama bias artabilir (Bias-Variance Tradeoff)

In [4]:
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score
df = sns.load_dataset("mpg")
df = df.dropna(subset=['horsepower']) # bağımsız değişkene ait null gözlemleri veri setinden çıkaralım.
# 1. Polynomial özellikleri oluştur (degree=3)
poly3 = PolynomialFeatures(degree=3)
X_poly3 = poly3.fit_transform(df[['horsepower']])
y = df['mpg']
# 2. Train/test ayır
X_train, X_test, y_train, y_test = train_test_split(X_poly3, y, test_size=0.2, random_state=42)
# 3. Scaling uygula (fit sadece train'e, transform her ikisine)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# 4. Farklı alpha değerleriyle Ridge dene
for alpha_deger in [0.01, 0.1, 1, 10, 100]:
    ridge_model = Ridge(alpha=alpha_deger)
    ridge_model.fit(X_train_scaled, y_train)
    y_pred = ridge_model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    print(f"Alpha={alpha_deger}: R²={r2:.4f}")

Alpha=0.01: R²=0.6388
Alpha=0.1: R²=0.6403
Alpha=1: R²=0.6423
Alpha=10: R²=0.6385
Alpha=100: R²=0.5465


### Ridge + Scaling Sonuçları ve Yorum

Scaling sonrası (StandardScaler ile), Ridge'in etkisi çok daha net ortaya çıktı:
- Alpha=0.01-0.1: R²~0.64, hafif overfitting hâlâ mevcut
- Alpha=1: R²=0.6423, en iyi denge noktası (cezasız modelin %63.83'ünden bile daha iyi)
- Alpha=10: R²=0.6385, hafif düşüş başlıyor
- Alpha=100: R²=0.5465, ciddi düşüş — ceza çok ağır, model artık underfitting'e kayıyor

Bu sonuç, alpha'nın (λ) çok küçük olmasının overfitting'i tam engelleyemediğini, çok büyük olmasının ise modeli gereğinden fazla basitleştirip underfitting'e sürüklediğini gösteriyor. Alpha=1 civarı, bu veri seti için bias-variance dengesinin en iyi olduğu nokta.

### Önemli Ders: Feature Scaling
Scaling öncesi Ridge'in etkisi neredeyse görünmüyordu (LinAlgWarning ile birlikte) çünkü x, x², x³ çok farklı ölçeklerdeydi ve ceza adil dağılmıyordu. Scaling sonrası Ridge'in gerçek etkisi ortaya çıktı. Bu, polynomial + regularization kullanılan her modelde scaling'in kritik bir ön adım olduğunu gösteriyor.